In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

In [3]:
from confs import *

In [4]:
# Create a SparkSession
# http://localhost:4040/


spark = SparkSession.builder \
    .appName("KafkaSparkParquet") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
24/11/16 21:09:51 WARN Utils: Your hostname, DM-LT05 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/11/16 21:09:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/test_env/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d8ba0104-344c-438f-9cbb-4fa4e45ff96c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 534ms :: artifacts dl 16ms
	:: m

In [5]:
# Define your schema (update it based on your Kafka data)
schema = StructType() \
    .add("device_id", StringType()) \
    .add("timestamp", DateType()) \
    .add("water_turbidity", FloatType())

In [15]:

# # Read data from Kafka in batch mode
# df = spark.read \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", KAFKA_BROKER) \
#     .option("subscribe", KAFKA_TOPIC) \
#     .load() \
#     .selectExpr("CAST(value AS STRING)") \
#     .select(from_json(col("value"), schema).alias("data")) \
#     .select("data.*")

# # Display the schema
# df.printSchema()

# # Show the Kafka DataFrame
# df.show(truncate=False)

In [ ]:
# Read data from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", KAFKA_TOPIC) \
    .load()

# Extract and parse the Kafka message
parsed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

# Write to Parquet in streaming mode
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", 'output/water-quality') \
    .option("checkpointLocation", "checkpoint") \
    .trigger(processingTime="2 minutes") \
    .start()

24/11/16 21:28:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


24/11/16 21:30:00 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [ ]:
query.stop()

In [ ]:
query.awaitTermination()